In [4]:
import numpy as np
import pandas as pd
import seaborn as sns


loading Titanic dataset

In [5]:
df = sns.load_dataset('titanic')

Step 1: Data Exploration

In [6]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [8]:
df.isnull().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [9]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns

print("Numerical:", numerical_cols)
print("Categorical:", categorical_cols)


Numerical: Index(['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare'], dtype='object')
Categorical: Index(['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='object')


In [10]:
df.nunique()


,0
survived,2
pclass,3
sex,2
age,88
sibsp,7
parch,7
fare,248
embarked,3
class,3
who,3


Step 2: Select Features

In [11]:
df = df.drop(columns=["alive", "embark_town", "class", "who"], errors='ignore')

In [12]:
X = df.drop("survived", axis=1)
y = df["survived"]

Step 3: Train/Test Split

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

Preprocessing + Model Training

Step 4: Identify Column Types

In [14]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category",  "bool"]).columns

Step 5: Handle Missing Values (Train Only)

In [15]:
from sklearn.impute import SimpleImputer
num_imputer = SimpleImputer(strategy="mean")
cat_imputer = SimpleImputer(strategy="most_frequent")
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

Step 6: Encoding & Scaling

In [16]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Model Training



Model 1: Logistic Regression

In [17]:
from sklearn.linear_model import LogisticRegression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

Model 2: Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Model 3: Support Vector Machine

In [19]:
from sklearn.svm import SVC
svm_model = SVC()
svm_model.fit(X_train, y_train)

SVC()

Evaluation

In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
models = {
    "Logistic Regression": log_model,
    "Random Forest": rf_model,
    "SVM": svm_model
}
for name, model in models.items():
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    print(f"\n{name}")
    print("Train Accuracy:", accuracy_score(y_train, train_preds))
    print("Test Accuracy:", accuracy_score(y_test, test_preds))
    print("Precision:", precision_score(y_test, test_preds))
    print("Recall:", recall_score(y_test, test_preds))
    print("F1 Score:", f1_score(y_test, test_preds))
    #For imbalanced dataset
    print("Weighted Precision:", precision_score(y_test, test_preds, average='weighted'))
    print("Weighted Recall:", recall_score(y_test, test_preds, average='weighted'))
    print("Weighted F1 Score:", f1_score(y_test, test_preds, average='weighted'))


Logistic Regression
Train Accuracy: 0.8370786516853933
Test Accuracy: 0.8156424581005587
Precision: 0.7808219178082192
Recall: 0.7702702702702703
F1 Score: 0.7755102040816326
Weighted Precision: 0.8153139624374234
Weighted Recall: 0.8156424581005587
Weighted F1 Score: 0.8154522578445448

Random Forest
Train Accuracy: 0.9845505617977528
Test Accuracy: 0.7932960893854749
Precision: 0.7605633802816901
Recall: 0.7297297297297297
F1 Score: 0.7448275862068966
Weighted Precision: 0.7923868474659252
Weighted Recall: 0.7932960893854749
Weighted F1 Score: 0.7926134344111287

SVM
Train Accuracy: 0.8356741573033708
Test Accuracy: 0.8156424581005587
Precision: 0.8059701492537313
Recall: 0.7297297297297297
F1 Score: 0.7659574468085106
Weighted Precision: 0.8150379387976319
Weighted Recall: 0.8156424581005587
Weighted F1 Score: 0.8140397158008151


In [21]:
# Import libraries
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Load dataset
df = sns.load_dataset("titanic")

# Drop irrelevant columns
df = df.drop(columns=["alive", "embark_town", "class", "who"])

# Separate features and target
X = df.drop("survived", axis=1)
y = df["survived"]

# Identify column types (BEFORE SPLIT → causes leakage)
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns

# ❌ WRONG: Handle missing values on entire dataset
num_imputer = SimpleImputer(strategy="mean")
cat_imputer = SimpleImputer(strategy="most_frequent")

X[num_cols] = num_imputer.fit_transform(X[num_cols])
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

# ❌ WRONG: Scale entire dataset
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# ❌ WRONG: Encode entire dataset
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# ❌ Split AFTER preprocessing → DATA LEAKAGE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train models
log_model = LogisticRegression(max_iter=1000)
rf_model = RandomForestClassifier(random_state=42)
svm_model = SVC()

models = {
    "Logistic Regression": log_model,
    "Random Forest": rf_model,
    "SVM": svm_model
}

for name, model in models.items():
    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    print(f"\n{name}")
    print("Train Accuracy:", accuracy_score(y_train, train_preds))
    print("Test Accuracy:", accuracy_score(y_test, test_preds))
    print("Precision:", precision_score(y_test, test_preds))
    print("Recall:", recall_score(y_test, test_preds))
    print("F1 Score:", f1_score(y_test, test_preds))



Logistic Regression
Train Accuracy: 0.8370786516853933
Test Accuracy: 0.8156424581005587
Precision: 0.7808219178082192
Recall: 0.7702702702702703
F1 Score: 0.7755102040816326

Random Forest
Train Accuracy: 0.9845505617977528
Test Accuracy: 0.7988826815642458
Precision: 0.7714285714285715
Recall: 0.7297297297297297
F1 Score: 0.75

SVM
Train Accuracy: 0.8356741573033708
Test Accuracy: 0.8156424581005587
Precision: 0.8059701492537313
Recall: 0.7297297297297297
F1 Score: 0.7659574468085106


Scikit-learn Pipeline

In [22]:
# Step 1: ColumnTransformer
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
transformers=[
("num", StandardScaler(), num_cols),
("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

In [25]:
#Step 2: Full Pipeline
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
("preprocess", preprocessor),
("model", LogisticRegression(max_iter=1000))])

In [28]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Re-load the dataset and perform initial preprocessing to ensure a clean state
df = sns.load_dataset('titanic')
df = df.drop(columns=["alive", "embark_town", "class", "who"], errors='ignore')

# Separate features and target
X = df.drop("survived", axis=1)
y = df["survived"]

# Perform train-test split on the raw data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify numerical and categorical columns from the raw X_train
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category",  "bool"]).columns

# Define preprocessing steps for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')) # drop='first' to avoid multicollinearity
])

# Create a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# Step 2: Create the full Pipeline with preprocessor and model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Step 3: Train & Evaluate the pipeline
pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)
print("Pipeline Accuracy:", accuracy_score(y_test, preds))

Pipeline Accuracy: 0.8156424581005587


In [30]:
#Step 4: Model Serialization (For Deployment Readiness)
# Once the pipeline is trained, save it for reuse:
import joblib
joblib.dump(pipeline, "titanic_pipeline.joblib")

['titanic_pipeline.joblib']

In [31]:
loaded_pipeline = joblib.load("titanic_pipeline.joblib")
preds = loaded_pipeline.predict(X_test)